# 00.5 · Verificación del entorno — `google/gemma-4-12B-it`

Este cuaderno se usa para asegurar que los modelos se comparen con versiones de librerias similares, y que estas sean soportadas en todas las etapas del trabajo.
1. Versiones, GPU y espacio en disco.
2. Plantilla de chat y máscara de pérdida (sin GPU).
3. Prueba de la evaluación base: carga en bf16 y genera 3 respuestas.
4. Prueba del ajuste fino: reinicia la memoria, carga en 4 bits + LoRA y ejecuta 2 pasos de `SFTTrainer`.
5. Guarda en `/workspace/verificacion/`:
   - `gemma4_12b_it.csv`: entorno (Python, torch, CUDA, cuDNN, GPU, driver y versión de cada librería), plantilla y tokens en la pérdida, resultados de las pruebas, memoria, tiempos y parámetros entrenables;
   - `gemma4_12b_it_pip_freeze.txt`: todas las librerías instaladas;
   - `gemma4_12b_it_nvidia_smi.txt`: estado de la GPU y del driver.

**Antes de empezar:** `requirements.txt` debe estar en `/workspace`. Se usa el kernel por defecto (**Python 3**).

## 0. Instalación del entorno

Se instalan las versiones fijadas sobre el entorno de la plantilla de RunPod (torch 2.8.0+cu128 y torchvision 0.23.0 ya vienen y no se descargan).

Para correr este cuaderno correctamente:

1. Se corre la celda de abajo (1–2 minutos).
2. **Se reinicia el kernel** (*Kernel → Restart*).
3. Se sigue desde la celda **1. Configuración**.

In [1]:
%pip install -q -r /workspace/requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 1. Configuración

In [1]:
NOMBRE = "gemma4_12b_it"
MODEL_ID = "google/gemma-4-12B-it"
REVISION = "707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7"
MODEL_TYPE = "multimodal"                 # "multimodal" o "causal_lm"
SUPPORTS_ENABLE_THINKING = True

## 2. Versiones, GPU y disco

In [2]:
import os
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
import importlib.metadata as md_, platform, shutil, torch

ESPERADAS = {
    "torch": "2.8.0", "torchvision": "0.23.0", "transformers": "5.15.0", "tokenizers": "0.22.2",
    "huggingface_hub": "1.27.0", "safetensors": "0.8.0", "Jinja2": "3.1.6", "pillow": "12.3.0",
    "sentencepiece": "0.2.2", "protobuf": "7.35.1", "peft": "0.20.0", "bitsandbytes": "0.50.1",
    "trl": "1.10.0", "accelerate": "1.14.0", "datasets": "5.0.1", "fsspec": "2026.6.0",
    "pyarrow": "25.0.1", "pandas": "3.0.5", "numpy": "2.5.2", "scikit-learn": "1.9.0",
    "matplotlib": "3.11.1", "seaborn": "0.13.2", "tqdm": "4.70.0", "ipywidgets": "8.1.8",
}
def _version(p):
    try: return md_.version(p)
    except md_.PackageNotFoundError: return "NO INSTALADA"
errores = [p for p, v in ESPERADAS.items() if _version(p).split("+")[0] != v]
assert not errores, f"Versiones distintas a las fijadas: {errores}"
print("Versiones OK | Python", platform.python_version(), "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())
import subprocess
DRIVER = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                        capture_output=True, text=True).stdout.strip()
ENTORNO = dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda,
               cudnn=torch.backends.cudnn.version(), gpu=torch.cuda.get_device_name(0),
               gpu_memoria_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), driver=DRIVER,
               **{f"v_{p}": _version(p) for p in ESPERADAS})
print("CUDA:", ENTORNO["cuda"], "| cuDNN:", ENTORNO["cudnn"], "| driver:", DRIVER)
assert ENTORNO["cuda"] == "12.8", f"CUDA distinta a la fijada: {ENTORNO['cuda']}"

total, usado, libre = shutil.disk_usage("/workspace")
print(f"Disco /workspace: {libre/1e9:.0f} GB libres de {total/1e9:.0f} GB")
if libre < 30e9:
    print("AVISO: menos de 30 GB libres; la descarga del modelo puede fallar (Disk quota exceeded)")

Versiones OK | Python 3.12.3 | torch 2.8.0+cu128
GPU: NVIDIA GeForce RTX 5090 | bf16: True
CUDA: 12.8 | cuDNN: 91002 | driver: 595.91.07
Disco /workspace: 401092 GB libres de 1036535 GB


In [3]:
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
assert hf_token, "Configura HF_TOKEN como variable de entorno del pod"
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 3. Plantilla de chat y máscara de pérdida (sin GPU)

In [4]:
from transformers import AutoTokenizer, AutoProcessor

if MODEL_TYPE == "multimodal":
    plantilla = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION, token=hf_token)
    tokenizer = plantilla.tokenizer
else:
    plantilla = tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, token=hf_token)

PROMPTS = {
    1: {"system": None,
        "user": "Clasifica el sentimiento del siguiente comentario. Responde solo con una de estas palabras: negativo, neutral, positivo. Comentario: {text}"},
    4: {"system": "Eres un clasificador de sentimientos de comentarios en espanol peruano. Responde unicamente con una de las siguientes clases: negativo, neutral, positivo.",
        "user": "Clasifica el siguiente comentario. Comentario: {text}"},
}
TEXTO, ETIQUETA = "Los de la U también son del Orto Equipo 😘", "negativo"
KW_THINK = {"enable_thinking": False} if SUPPORTS_ENABLE_THINKING else {}

def construir_mensajes(prompt_id, texto):
    c = PROMPTS[prompt_id]
    m = [{"role": "system", "content": c["system"]}] if c["system"] else []
    return m + [{"role": "user", "content": c["user"].format(text=texto)}]

def ejemplo_tokenizado(prompt_id, texto, etiqueta):
    m_p = construir_mensajes(prompt_id, texto)
    prompt_txt = plantilla.apply_chat_template(m_p, add_generation_prompt=True, tokenize=False, **KW_THINK)
    full_txt = plantilla.apply_chat_template(m_p + [{"role": "assistant", "content": etiqueta}], tokenize=False)
    completion_txt = full_txt[full_txt.rfind(etiqueta):]
    p_ids = tokenizer(prompt_txt, add_special_tokens=False)["input_ids"]
    c_ids = tokenizer(completion_txt, add_special_tokens=False)["input_ids"]
    return {"input_ids": p_ids + c_ids, "labels": [-100] * len(p_ids) + c_ids}

REGISTRO_PLANTILLA = {}
for pid in (1, 4):
    ej = ejemplo_tokenizado(pid, TEXTO, ETIQUETA)
    en = [t for t, l in zip(ej["input_ids"], ej["labels"]) if l != -100]
    print(f"Prompt {pid}")
    print("  Secuencia completa:", repr(tokenizer.decode(ej["input_ids"])))
    print(f"  En la perdida: {len(en)} de {len(ej['input_ids'])} -> {tokenizer.decode(en)!r}")
    REGISTRO_PLANTILLA[f"p{pid}_secuencia"] = tokenizer.decode(ej["input_ids"])
    REGISTRO_PLANTILLA[f"p{pid}_en_perdida"] = tokenizer.decode(en)
    REGISTRO_PLANTILLA[f"p{pid}_tokens_perdida"] = f"{len(en)}/{len(ej['input_ids'])}"

processor_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.09k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

Prompt 1
  Secuencia completa: '<bos><|turn>user\nClasifica el sentimiento del siguiente comentario. Responde solo con una de estas palabras: negativo, neutral, positivo. Comentario: Los de la U también son del Orto Equipo 😘<turn|>\n<|turn>model\n<|channel>thought\n<channel|>negativo<turn|>\n'
  En la perdida: 4 de 55 -> 'negativo<turn|>\n'
Prompt 4
  Secuencia completa: '<bos><|turn>system\nEres un clasificador de sentimientos de comentarios en espanol peruano. Responde unicamente con una de las siguientes clases: negativo, neutral, positivo.<turn|>\n<|turn>user\nClasifica el siguiente comentario. Comentario: Los de la U también son del Orto Equipo 😘<turn|>\n<|turn>model\n<|channel>thought\n<channel|>negativo<turn|>\n'
  En la perdida: 4 de 74 -> 'negativo<turn|>\n'


## 4. Prueba de la evaluación base (bf16)

In [5]:
import gc, time, pandas as pd
from transformers import AutoModelForCausalLM, AutoModelForMultimodalLM
ClaseModelo = AutoModelForMultimodalLM if MODEL_TYPE == "multimodal" else AutoModelForCausalLM
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
resultado = dict(modelo=MODEL_ID, revision=REVISION, fecha=time.strftime("%Y-%m-%d %H:%M"), **ENTORNO, **REGISTRO_PLANTILLA)

torch.cuda.reset_peak_memory_stats(); t0 = time.time()
model = ClaseModelo.from_pretrained(MODEL_ID, revision=REVISION, token=hf_token,
                                    dtype=torch.bfloat16, device_map="auto")
salidas = []
for t in [TEXTO, "Qué buena chamba la de hoy, todo salió chévere", "Mañana hay reunión a las 10"]:
    p = plantilla.apply_chat_template(construir_mensajes(1, t), add_generation_prompt=True, tokenize=False, **KW_THINK)
    ids = tokenizer(p, add_special_tokens=False, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=10, do_sample=False, pad_token_id=PAD_ID)
    salidas.append(tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))
resultado.update(base_salidas=salidas, base_mem_gb=round(torch.cuda.max_memory_allocated()/1e9, 2),
                 base_seg=round(time.time() - t0))
print(resultado)

del model; gc.collect(); torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B / 23.9GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

{'modelo': 'google/gemma-4-12B-it', 'revision': '707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7', 'fecha': '2026-10-24 23:06', 'python': '3.12.3', 'torch': '2.8.0+cu128', 'cuda': '12.8', 'cudnn': 91002, 'gpu': 'NVIDIA GeForce RTX 5090', 'gpu_memoria_gb': 33.7, 'driver': '595.91.07', 'v_torch': '2.8.0+cu128', 'v_torchvision': '0.23.0+cu128', 'v_transformers': '5.15.0', 'v_tokenizers': '0.22.2', 'v_huggingface_hub': '1.27.0', 'v_safetensors': '0.8.0', 'v_Jinja2': '3.1.6', 'v_pillow': '12.3.0', 'v_sentencepiece': '0.2.2', 'v_protobuf': '7.35.1', 'v_peft': '0.20.0', 'v_bitsandbytes': '0.50.1', 'v_trl': '1.10.0', 'v_accelerate': '1.14.0', 'v_datasets': '5.0.1', 'v_fsspec': '2026.6.0', 'v_pyarrow': '25.0.1', 'v_pandas': '3.0.5', 'v_numpy': '2.5.2', 'v_scikit-learn': '1.9.0', 'v_matplotlib': '3.11.1', 'v_seaborn': '0.13.2', 'v_tqdm': '4.70.0', 'v_ipywidgets': '8.1.8', 'p1_secuencia': '<bos><|turn>user\nClasifica el sentimiento del siguiente comentario. Responde solo con una de estas palabras: negat

## 5. Prueba del ajuste fino (QLoRA, 2 pasos)

In [6]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch.cuda.reset_peak_memory_stats(); t0 = time.time()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = ClaseModelo.from_pretrained(MODEL_ID, revision=REVISION, token=hf_token,
                                    quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()
entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
totales = sum(p.numel() for p in model.parameters())

ds = Dataset.from_list([ejemplo_tokenizado(1, TEXTO, ETIQUETA)] * 16)
args = SFTConfig(output_dir=f"/workspace/smoke_{NOMBRE}", max_steps=2, per_device_train_batch_size=8,
                 learning_rate=2e-4, bf16=True, logging_steps=1, report_to="none", save_strategy="no",
                 max_length=512, packing=False, gradient_checkpointing=False)
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, processing_class=tokenizer)
res = trainer.train()
resultado.update(qlora_param_entrenables=entrenables, qlora_param_totales=totales,
                 qlora_pct_entrenables=round(100 * entrenables / totales, 4), qlora_loss=round(res.training_loss, 4), qlora_mem_gb=round(torch.cuda.max_memory_allocated()/1e9, 2),
                 qlora_seg=round(time.time() - t0))
print(resultado)

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

trainable params: 65,568,768 || all params: 12,025,298,944 || trainable%: 0.5453


Truncating train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,6.248882
2,2.924817


{'modelo': 'google/gemma-4-12B-it', 'revision': '707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7', 'fecha': '2026-10-24 23:06', 'python': '3.12.3', 'torch': '2.8.0+cu128', 'cuda': '12.8', 'cudnn': 91002, 'gpu': 'NVIDIA GeForce RTX 5090', 'gpu_memoria_gb': 33.7, 'driver': '595.91.07', 'v_torch': '2.8.0+cu128', 'v_torchvision': '0.23.0+cu128', 'v_transformers': '5.15.0', 'v_tokenizers': '0.22.2', 'v_huggingface_hub': '1.27.0', 'v_safetensors': '0.8.0', 'v_Jinja2': '3.1.6', 'v_pillow': '12.3.0', 'v_sentencepiece': '0.2.2', 'v_protobuf': '7.35.1', 'v_peft': '0.20.0', 'v_bitsandbytes': '0.50.1', 'v_trl': '1.10.0', 'v_accelerate': '1.14.0', 'v_datasets': '5.0.1', 'v_fsspec': '2026.6.0', 'v_pyarrow': '25.0.1', 'v_pandas': '3.0.5', 'v_numpy': '2.5.2', 'v_scikit-learn': '1.9.0', 'v_matplotlib': '3.11.1', 'v_seaborn': '0.13.2', 'v_tqdm': '4.70.0', 'v_ipywidgets': '8.1.8', 'p1_secuencia': '<bos><|turn>user\nClasifica el sentimiento del siguiente comentario. Responde solo con una de estas palabras: negat

## 6. Guardar resultado

In [7]:
os.makedirs("/workspace/verificacion", exist_ok=True)
pd.DataFrame([resultado]).to_csv(f"/workspace/verificacion/{NOMBRE}.csv", index=False)
subprocess.run(f"pip freeze > /workspace/verificacion/{NOMBRE}_pip_freeze.txt", shell=True)
subprocess.run(f"nvidia-smi > /workspace/verificacion/{NOMBRE}_nvidia_smi.txt", shell=True)
import shutil as _sh; _sh.rmtree(f"/workspace/smoke_{NOMBRE}", ignore_errors=True)
print("Guardado en /workspace/verificacion/:", sorted(f for f in os.listdir("/workspace/verificacion") if f.startswith(NOMBRE)))
pd.DataFrame([resultado]).T

Guardado en /workspace/verificacion/: ['gemma4_12b_it.csv', 'gemma4_12b_it_nvidia_smi.txt', 'gemma4_12b_it_pip_freeze.txt']


,0
modelo,google/gemma-4-12B-it
revision,707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7
fecha,2026-10-24 23:06
python,3.12.3
torch,2.8.0+cu128
cuda,12.8
cudnn,91002
gpu,NVIDIA GeForce RTX 5090
gpu_memoria_gb,33.7
driver,595.91.07
